# Lab 2 — Deep Learning 2026 (Student version)

In this practical you will complete the missing parts of the code in three short experiments:

1. Gradient descent on nonlinear data.
2. Learning-rate behaviour of the Adam optimiser on MNIST.
3. CNNs and generalisation on Fashion-MNIST.

**How to run:** Open in Google Colab, choose **Runtime → Change runtime type → T4 GPU** if available, then run the cells from top to bottom. The notebook also runs on CPU, only more slowly.

### How this notebook works
- Cells marked **`# TODO`** or **`# <-- YOUR CODE HERE`** need your code. Until you fill them in, they will raise an error or fail a check — that is expected.
- **Questions** are for you to answer in the text cell below them.

The experiments use small subsets and short runs to demonstrate concepts quickly, not to reach state-of-the-art accuracy.

## 0. Setup (nothing to change)

This cell selects a GPU when available, fixes random seeds for reproducibility, and imports the libraries.

In [ ]:
import copy
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

# Exercise 1 — Gradient descent and nonlinear regression (≈5 minutes)

We generate noisy observations from $y = x^2$. A straight line cannot represent this relationship, so we compare:

- **Linear model:** $\hat y = wx + b$
- **Polynomial model:** $\hat y = ax^2 + bx + c$

**Your tasks**
1. Build the feature matrix `[x, x², 1]` for the quadratic model.
2. Write the four lines of a gradient-descent training step.

The key idea: *optimisation and model choice are separate things.* Gradient descent finds good parameters **for the model you chose** — it cannot turn a linear model into a quadratic one.

The data cell below is provided.

In [ ]:
# Reproducible synthetic dataset (provided - do not change)
g = torch.Generator().manual_seed(SEED)
x = torch.linspace(-2, 2, 800).view(-1, 1)          # shape (800, 1)
y_true = x**2
y = y_true + 0.20 * torch.randn(x.shape, generator=g)  # add Gaussian noise

# Fixed train / validation split
perm = torch.randperm(len(x), generator=g)
train_idx, val_idx = perm[:640], perm[640:]
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

# Mini-batches of 64 examples, reshuffled every epoch
train_loader_reg = DataLoader(
    TensorDataset(x_train, y_train), batch_size=64, shuffle=True
)

### Task 1.1 — Define the two models

`LinearRegressor` is given. Complete `QuadraticRegressor.forward`.

In [ ]:
class LinearRegressor(nn.Module):
    """y_hat = w*x + b  (one input feature)"""
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    def forward(self, x):
        return self.linear(x)


class QuadraticRegressor(nn.Module):
    """y_hat = a*x^2 + b*x + c, implemented as a LINEAR layer on polynomial features.

    The model is still linear in its PARAMETERS - only the inputs are transformed.
    """
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 1)   # 3 input features: x, x^2, 1

    def forward(self, x):
        # TODO: build a feature matrix of shape (N, 3) whose columns are [x, x**2, 1]
        features = None  # <-- YOUR CODE HERE
        return self.linear(features)

### Task 1.2 — Write the training step

Inside the loop, every mini-batch needs the standard four-step PyTorch update. They must be in the right **order**:

| Step | What it does |
|---|---|
| clear old gradients | gradients accumulate in PyTorch, so reset them |
| forward pass + loss | compute predictions, then mean squared error |
| backward pass | compute ∂loss/∂parameters |
| optimiser step | move parameters a small step downhill |

In [ ]:
def fit_regressor(model, epochs=100):
    optimiser = torch.optim.SGD(model.parameters(), lr=0.05)  # plain gradient descent
    losses = []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader_reg:
            # TODO: the four steps of one gradient-descent update
            # 1) reset gradients                        (optimiser.????)
            # 2) compute the MSE loss of model(xb) vs yb (F.mse_loss)
            # 3) back-propagate                          (loss.????)
            # 4) update the parameters                   (optimiser.????)
            loss = None  # <-- YOUR CODE HERE (replace with all four steps; keep the name `loss`)

            running += loss.item() * len(xb)   # accumulate loss for reporting
        losses.append(running / len(train_loader_reg.dataset))

    # Evaluate on held-out validation data (no gradients needed)
    model.eval()
    with torch.no_grad():
        val_loss = F.mse_loss(model(x_val), y_val).item()
    return losses, val_loss

### Train both models (provided)

In [ ]:
linear_model = LinearRegressor()
quadratic_model = QuadraticRegressor()
linear_loss, linear_val = fit_regressor(linear_model)
quadratic_loss, quadratic_val = fit_regressor(quadratic_model)

print(f"Linear model validation MSE:    {linear_val:.4f}")
print(f"Quadratic model validation MSE: {quadratic_val:.4f}")

# Check
assert quadratic_val < linear_val, "The quadratic model should beat the linear one - re-check your code."
assert quadratic_val < 0.1, "Quadratic MSE should be close to the noise level (0.2^2 = 0.04)."
print("Exercise 1 checks passed")

In [ ]:
# Plot the fits and training losses (provided)
x_plot = torch.linspace(-2, 2, 300).view(-1, 1)
with torch.no_grad():
    y_linear = linear_model(x_plot)
    y_quadratic = quadratic_model(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x.numpy(), y.numpy(), s=8, alpha=0.25, label="Noisy observations")
axes[0].plot(x_plot.numpy(), x_plot.pow(2).numpy(), "k--", label="True function")
axes[0].plot(x_plot.numpy(), y_linear.numpy(), label="Linear model")
axes[0].plot(x_plot.numpy(), y_quadratic.numpy(), label="Quadratic model")
axes[0].set(title="Model fit", xlabel="x", ylabel="y")
axes[0].legend(); axes[0].grid(alpha=0.25)

axes[1].plot(linear_loss, label="Linear")
axes[1].plot(quadratic_loss, label="Quadratic")
axes[1].set(title="Training loss", xlabel="Epoch", ylabel="MSE")
axes[1].legend(); axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

### Questions 1

1. The linear model's loss flattens out at a high value. Would training it for 1,000 more epochs fix this? Why / why not?
2. The quadratic model is still "linear regression". In what sense is it linear?

**Your answers:**

*(double-click to write here)*

**Optional extension (1 min):** change the target to `x**3`, add an `x**3` column to the features (and make `nn.Linear` take 4 inputs). Does the fit still work?

# Exercise 2 — Learning rates on MNIST

We train the **same** small multilayer perceptron (MLP) with three different Adam learning rates and compare them on a **validation** set. The official test set is *not* used for tuning.

**Your tasks**
1. Define the MLP layers.
2. Write the training step for classification.
3. Pick the best learning rate from the results.

The data loading below is provided. We use 8,000 training and 2,000 validation images to keep things fast.

In [ ]:
# MNIST: split the original training data into train and validation subsets (provided)
mnist_transform = transforms.ToTensor()   # converts images to tensors with values in [0, 1]
mnist_full = datasets.MNIST("./data", train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST("./data", train=False, download=True, transform=mnist_transform)

mnist_train_subset, mnist_val_subset, _ = random_split(
    mnist_full, [8000, 2000, len(mnist_full) - 10000],
    generator=torch.Generator().manual_seed(SEED)
)
mnist_train_loader = DataLoader(mnist_train_subset, batch_size=256, shuffle=True, num_workers=0)
mnist_val_loader = DataLoader(mnist_val_subset, batch_size=512, shuffle=False, num_workers=0)

images, labels = next(iter(mnist_train_loader))
print("Batch of images:", images.shape, "| labels:", labels.shape)   # (256, 1, 28, 28)

### Task 2.1 — Define the MLP

Build a network with this structure:

`Flatten → Linear(784 → 128) → ReLU → Linear(128 → 10)`

Each MNIST image is 28×28 = 784 pixels, and there are 10 digit classes. The output is 10 raw scores (**logits**) — do **not** add a softmax; `F.cross_entropy` applies it internally.

In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # TODO: fill in the four layers listed above, separated by commas
            # <-- YOUR CODE HERE
        )
    def forward(self, x):
        return self.net(x)

# Provided helper: percentage of correctly classified examples in a data loader
def evaluate_accuracy(model, loader, device):
    model.eval()                      # switch off dropout / use running BN statistics
    correct, total = 0, 0
    with torch.no_grad():             # no gradients needed for evaluation
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)   # class with the highest logit
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

# Check the architecture produces 10 logits per image
_m = MNISTMLP()
assert _m(torch.zeros(5, 1, 28, 28)).shape == (5, 10), "Output should have shape (batch, 10)"
print("MLP output shape OK")

### Task 2.2 — Learning-rate experiment

For each learning rate we **restore the same initial weights**, so any difference in the results comes from the learning rate alone (a *controlled experiment*).

Complete the training step inside the loop. It is the same four-step pattern as Exercise 1, but with **cross-entropy** loss (the standard loss for classification).

In [ ]:
torch.manual_seed(SEED)
mnist_model = MNISTMLP().to(device)
initial_state = copy.deepcopy(mnist_model.state_dict())   # save the starting weights

learning_rates = [1e-4, 1e-3, 1e-2]
lr_results = []

for lr in learning_rates:
    mnist_model.load_state_dict(initial_state)                      # identical start every time
    optimiser = torch.optim.Adam(mnist_model.parameters(), lr=lr)   # new optimiser per learning rate
    epoch_losses, val_accuracies = [], []
    start = time.time()

    for epoch in range(2):
        mnist_model.train()
        total_loss, seen = 0.0, 0
        for images, labels in mnist_train_loader:
            images, labels = images.to(device), labels.to(device)

            # TODO: one training step
            #   1) zero the gradients
            #   2) logits = model output for `images`; loss = cross-entropy(logits, labels)
            #   3) backward pass
            #   4) optimiser step
            loss = None  # <-- YOUR CODE HERE (keep the name `loss`)

            total_loss += loss.item() * labels.size(0)   # weight by batch size
            seen += labels.size(0)

        epoch_losses.append(total_loss / seen)
        val_accuracies.append(evaluate_accuracy(mnist_model, mnist_val_loader, device))

    lr_results.append({
        "lr": lr, "losses": epoch_losses,
        "val_acc": val_accuracies, "seconds": time.time() - start
    })
    print(f"Adam lr={lr:g}: validation accuracy={val_accuracies[-1]:.1f}% "
          f"(elapsed {lr_results[-1]['seconds']:.1f}s)")

### Task 2.3 — Plot and choose the best learning rate

Plots are provided. You only need to select the result with the highest **final validation accuracy**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for result in lr_results:
    label = f"lr={result['lr']:g}"
    axes[0].plot(result["losses"], marker="o", label=label)
    axes[1].plot(result["val_acc"], marker="o", label=label)
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
for ax in axes:
    ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.show()

# TODO: pick the dictionary in lr_results with the highest FINAL validation accuracy
# (the final value is r["val_acc"][-1])
best_result = None  # <-- YOUR CODE HERE
print("Best learning rate in this short run:", best_result["lr"])


### Questions 2

1. Describe what happened with the smallest learning rate (1e-4). Why?
2. Which learning rate worked best for you? Would you expect that to be the best for *every* problem?
3. Why do we choose the learning rate using **validation** accuracy rather than **test** accuracy?

**Your answers:**

*(double-click to write here)*

# Exercise 3 — CNNs and generalisation on Fashion-MNIST

Fashion-MNIST has 28×28 greyscale images of 10 clothing categories. We compare:

- a compact **baseline CNN** (provided), and
- an **improved CNN** that adds *batch normalisation* and *dropout* (you will build it).

Both get the same data and training budget: 10,000 training images, 2,000 validation images, 3 epochs. The 10,000-image **test set is used only once, at the end**.

**Your tasks**
1. Complete the improved CNN.
2. Write the training step with accuracy tracking.
3. Find misclassified test images.

In [ ]:
# Data (provided)
fashion_transform = transforms.ToTensor()
fashion_full = datasets.FashionMNIST("./data", train=True, download=True, transform=fashion_transform)
fashion_test = datasets.FashionMNIST("./data", train=False, download=True, transform=fashion_transform)

fashion_train, fashion_val, _ = random_split(
    fashion_full, [10000, 2000, len(fashion_full) - 12000],
    generator=torch.Generator().manual_seed(SEED)
)
fashion_train_loader = DataLoader(fashion_train, batch_size=128, shuffle=True, num_workers=0)
fashion_val_loader = DataLoader(fashion_val, batch_size=512, shuffle=False, num_workers=0)
fashion_test_loader = DataLoader(fashion_test, batch_size=512, shuffle=False, num_workers=0)

print("Class names:", fashion_full.classes)

### Task 3.1 — Models

The baseline CNN is given. Study it first: after each `MaxPool2d(2)` the image size halves (28 → 14 → 7), which is why the classifier starts with `32 * 7 * 7` inputs.

Then complete `ImprovedCNN`:

| Block | Layers |
|---|---|
| conv block 1 | `Conv2d(1→32, 3, padding=1, bias=False)` → `BatchNorm2d(32)` → `ReLU` → `MaxPool2d(2)` |
| conv block 2 | `Conv2d(32→64, 3, padding=1, bias=False)` → `BatchNorm2d(64)` → `ReLU` → `MaxPool2d(2)` |
| classifier | `Flatten` → `Linear(64*7*7 → 128)` → `ReLU` → `Dropout(0.30)` → `Linear(128 → 10)` |

(`bias=False` in the convolutions because BatchNorm already has its own shift parameter.)

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 1x28x28 -> 16x28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                               # -> 16x14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),   # -> 32x14x14
            nn.ReLU(),
            nn.MaxPool2d(2)                                # -> 32x7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # TODO: conv block 1 and conv block 2 from the table above
            # <-- YOUR CODE HERE (8 layers in total, separated by commas)
        )
        self.classifier = nn.Sequential(
            # TODO: the classifier layers from the table above
            # <-- YOUR CODE HERE (5 layers)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# Check
_m = ImprovedCNN()
assert _m(torch.zeros(4, 1, 28, 28)).shape == (4, 10), "Output should have shape (batch, 10)"
print("ImprovedCNN output shape OK | parameters:", sum(p.numel() for p in _m.parameters()))
print("   BaselineCNN parameters:", sum(p.numel() for p in BaselineCNN().parameters()))

### Task 3.2 — Training function

This time you also track **training accuracy**, so you can compare it with validation accuracy and spot overfitting.

Inside the batch loop you need to:
1. do the four-step update (as before),
2. add the batch loss (weighted by batch size) to `total_loss`,
3. count how many predictions were correct,
4. count how many examples were seen.

In [ ]:
def train_classifier(model, train_loader, val_loader, epochs=3, lr=1e-3):
    model = model.to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()   # IMPORTANT: enables dropout and batch-norm training behaviour
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # TODO 1: zero gradients, compute logits and cross-entropy loss, backward, optimiser step
            logits = None  # <-- YOUR CODE HERE (must define both `logits` and `loss`)

            # TODO 2: update the running statistics
            #   total_loss += (batch loss as a Python float) * (batch size)
            #   correct    += number of examples where argmax(logits) == labels
            #   total      += batch size
            # <-- YOUR CODE HERE

        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(100 * correct / total)
        history["val_acc"].append(evaluate_accuracy(model, val_loader, device))
        print(f"Epoch {epoch+1}/{epochs} | loss {history['train_loss'][-1]:.3f} "
              f"| train acc {history['train_acc'][-1]:.1f}% "
              f"| val acc {history['val_acc'][-1]:.1f}%")
    return model, history

### Train both models (provided)

In [ ]:
print("Training baseline CNN")
baseline_model, baseline_history = train_classifier(
    BaselineCNN(), fashion_train_loader, fashion_val_loader, epochs=3
)
print("\nTraining improved CNN")
improved_model, improved_history = train_classifier(
    ImprovedCNN(), fashion_train_loader, fashion_val_loader, epochs=3
)

# Check
assert baseline_history["val_acc"][-1] > 70 and improved_history["val_acc"][-1] > 70, \
    "Validation accuracy should be well above 70% - re-check your training step."
print("\n Training checks passed")

In [ ]:
# Compare the two models (provided)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, history in [("Baseline", baseline_history), ("Improved", improved_history)]:
    axes[0].plot(history["train_loss"], marker="o", label=name)
    axes[1].plot(history["val_acc"], marker="o", label=name)
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
for ax in axes:
    ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.show()

# The test set is used ONLY now, once, after all choices are made
baseline_test_acc = evaluate_accuracy(baseline_model, fashion_test_loader, device)
improved_test_acc = evaluate_accuracy(improved_model, fashion_test_loader, device)
print(f"Baseline CNN test accuracy: {baseline_test_acc:.2f}%")
print(f"Improved CNN test accuracy: {improved_test_acc:.2f}%")

### Task 3.3 — Inspect the model's mistakes

Accuracy alone does not tell you *what* a classifier gets wrong. Collect up to 8 misclassified test images from the improved model.

Complete the line that creates a boolean mask `wrong` that is `True` where the prediction differs from the true label.

In [ ]:
class_names = fashion_full.classes
improved_model.eval()
wrong_images, wrong_true, wrong_pred = [], [], []

with torch.no_grad():
    for images, labels in fashion_test_loader:
        images_device = images.to(device)
        predictions = improved_model(images_device).argmax(1).cpu()   # back to CPU for indexing

        # TODO: boolean mask, True where the prediction is WRONG
        wrong = None  # <-- YOUR CODE HERE

        for image, true_label, pred_label in zip(images[wrong], labels[wrong], predictions[wrong]):
            wrong_images.append(image.squeeze(0))      # (1,28,28) -> (28,28) for plotting
            wrong_true.append(true_label.item())
            wrong_pred.append(pred_label.item())
            if len(wrong_images) == 8:
                break
        if len(wrong_images) == 8:
            break

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, true_label, pred_label in zip(axes.flat, wrong_images, wrong_true, wrong_pred):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Examples misclassified by the improved CNN")
plt.tight_layout(); plt.show()

### Questions 3

1. Did the improved CNN beat the baseline? Is the difference large enough to be sure, given only 3 epochs on a subset and one random seed?
2. Which classes are confused most often in the mistakes above? Why might they look alike?
3. What do **batch normalisation** and **dropout** each do? Which one is only active during training?
4. If training accuracy keeps rising but validation accuracy stops improving, what is probably happening, and name two things you could try.

**Your answers:**

*(double-click to write here)*

# Wrap-up

| Concept | Main takeaway |
|---|---|
| Model capacity | A model must have a suitable form to represent the data. |
| Gradient descent | Optimisation updates parameters to reduce a loss function. |
| Learning rate | Controls the size of each optimisation step. |
| Validation set | Lets you choose settings without repeatedly using the test set. |
| CNN | Learns spatial features from images using convolutional filters. |
| Regularisation | Can improve generalisation when a model overfits. |

**Going further (optional):** try `lr=3e-3` for the improved CNN; train for 10 epochs and plot train vs. validation accuracy; add data augmentation (e.g. `transforms.RandomHorizontalFlip()`) to the training transform.